# Coverless Steganography — Full Pipeline Visualization & Demo

This notebook walks through every stage of the multi-object coverless steganography pipeline,
with **actual images**, **live attacks**, and an **interactive upload demo** where you play
the role of the receiver.

**Sections:**
1. Setup & Data Loading
2. Pipeline Overview Diagram
3. Object Detection & Dictionary
4. Message Hiding (Sender Side)
5. Attack Gallery
6. Retrieval & Decoding (Receiver Side)
7. Interactive Upload Demo
8. Phase 2 vs Phase 3 Results Comparison
9. Stress-Testing the 100% Claim

**Requirements:** GPU T4 (for Faster R-CNN + SSCD). Attach the Phase 3 notebook output as input
(it contains extraction caches and SSCD weights, so no re-extraction is needed).

---
## Section 0 — Settings

Change only these values. Everything else adapts.

In [ ]:
# ===== SETTINGS =====
REPO = "i-git-abhishek/Enhanced-Coverless-Steganography"
BRANCH = "rewrite-baseline"
CONFIG = "configs/voc2012.yaml"

# The secret message for the demo. Change it to anything you like!
SECRET_MESSAGE = "HELLO"

# Number of sample images to show in the detection/hiding walkthrough
NUM_SAMPLE_IMAGES = 6

# Attack to use for the detailed retrieval walkthrough
DEMO_ATTACK = "Rotation 30°"

---
## Section 1 — Setup & Data Loading

Clone the repository, copy cached data from attached datasets, and load the
VOC2012 images + pre-computed caches (extraction, clusters, SSCD descriptors).

### 1a — Clone repo & check GPU

In [ ]:
"""Clone the repo and verify the GPU is available."""
import os, subprocess, sys

def run(cmd):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True)

# Clone
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

folder = "/kaggle/working/" + REPO.split("/")[1]
if not os.path.isdir(folder):
    auth = f"x-access-token:{token}@" if token else ""
    result = subprocess.run(
        ["git", "clone", "--quiet", "--branch", BRANCH,
         f"https://{auth}github.com/{REPO}.git", folder],
        capture_output=True
    )
    if result.returncode != 0:
        raise SystemExit("Clone failed. Check REPO and BRANCH.")
    subprocess.run(["git", "-C", folder, "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)

os.chdir(folder)
run("git log -1 --oneline")
subprocess.run("nvidia-smi", shell=True)
print("\nWorking directory:", os.getcwd())

### 1b — Copy caches from attached datasets

The Phase 3 notebook output (attached as a Kaggle dataset) contains:
- Extraction cache (Faster R-CNN detections + ResNet-101 features)
- KMeans clusters
- SSCD descriptors
- Retrievability check results
- SSCD weights file
- Experiment run results (for Phase 2 vs Phase 3 comparison)

We copy all of it so nothing needs to be re-computed.

In [ ]:
"""Copy caches and weights from attached input datasets."""
import glob, shutil

# Copy all finished caches
for done in glob.glob("/kaggle/input/**/cache/*/COMPLETE", recursive=True):
    source = os.path.dirname(done)
    target = os.path.join("cache", os.path.basename(source))
    if not os.path.exists(target):
        shutil.copytree(source, target)
        print(f"Copied cache: {os.path.basename(source)}")

# Copy SSCD weights
for weights in glob.glob("/kaggle/input/**/sscd_disc_mixup.torchscript.pt", recursive=True):
    os.makedirs("cache/weights", exist_ok=True)
    dest = "cache/weights/sscd_disc_mixup.torchscript.pt"
    if not os.path.exists(dest):
        shutil.copy(weights, dest)
        print("Copied SSCD weights")
    break

# Copy experiment runs (for Phase 2 vs 3 comparison)
for summary in glob.glob("/kaggle/input/**/experiments/runs/*/summary.json", recursive=True):
    run_dir = os.path.dirname(summary)
    run_name = os.path.basename(run_dir)
    target = os.path.join("experiments", "runs", run_name)
    if not os.path.exists(target):
        shutil.copytree(run_dir, target)
        print(f"Copied run: {run_name}")

print("\nCaches available:", sorted(os.listdir("cache")) if os.path.isdir("cache") else "none")
if os.path.isdir("experiments/runs"):
    print("Experiment runs:", sorted(os.listdir("experiments/runs")))

### 1c — Load the pipeline modules

This cell imports everything we need from the `stego` package and loads
the dataset (image paths), the cached extraction (detections + features),
clusters, SSCD descriptors, and builds the image index.

In [ ]:
"""Import the stego package and load all cached data."""
import json
import warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, HTML, Markdown

# Make plots inline and high-resolution
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['savefig.dpi'] = 150
matplotlib.rcParams['figure.facecolor'] = 'white'
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore', category=FutureWarning)

# Our modules
from stego.config import load_config
from stego.data import discover_images
from stego.cache import load_extraction
from stego.pipeline import resolve_paths, load_clusters_or_build, ordered_attacks
from stego.mapping import (
    build_index, build_dictionary, label_image_counts, top_objects, segment_to_bits
)
from stego.hiding import bytes_to_segments, segments_to_bytes, hide_segments, StegoImage
from stego.recovery import (
    read_image, decode_image, make_retriever, DescriptorRetriever
)
from stego.attacks import (
    build_attacks, attack_rng, NO_ATTACK, TABLE_IX_ORDER,
    GEOMETRIC, NON_GEOMETRIC, COMBINED, HEAVY, EXTENDED_ORDER
)
from stego.descriptors import DescriptorModel, load_descriptors

print("All modules loaded.")

### 1d — Load dataset and build the image index

In [ ]:
"""Load config, discover images, load caches, build index."""
config = load_config(CONFIG)
paths = resolve_paths(None, "cache", "experiments/runs", [])

# Discover all images
ids, image_paths = discover_images(config, paths.data_root)
print(f"Dataset: {config.dataset}")
print(f"Total images: {len(ids)}")

# Load cached extraction (Faster R-CNN detections + ResNet-101 features)
extraction = load_extraction(ids, config.dataset, paths.cache_dir, paths.read_dirs)
print(f"Extraction loaded: {len(extraction.ids)} images, features shape {extraction.features.shape}")

# Load clusters
clusters = load_clusters_or_build(config, extraction, paths)
print(f"Clusters: {config.k} groups")

# Build the object→group dictionary and image index
counts = label_image_counts(extraction.objects, config.conf_threshold, config.M)
index = build_index(
    ids, extraction.objects, clusters,
    config.n, config.M, config.conf_threshold
)
print(f"Dictionary: {len(index.dictionary)} labels → {index.groups} groups (2^{config.n})")
print(f"Usable images (with ≥1 object above threshold): {index.usable_images}/{len(ids)}")

# Load SSCD descriptors and model
descriptors = load_descriptors(ids, config.dataset, "sscd_disc_mixup", paths.cache_dir, paths.read_dirs)
sscd_model = DescriptorModel("sscd_disc_mixup", weights_dir=Path("cache/weights"))
print(f"SSCD descriptors loaded: {descriptors.shape}, device: {sscd_model.device}")

### 1e — Helper: image display utility

A reusable function for showing images in grids throughout the notebook.
OpenCV loads images as BGR; this converts them to RGB for matplotlib.

In [ ]:
"""Reusable display helpers used throughout the notebook."""

def bgr_to_rgb(image):
    """Convert OpenCV BGR to RGB for matplotlib."""
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def show_images(images, titles=None, cols=3, figsize=None, suptitle=None):
    """Display a list of BGR images in a grid."""
    n = len(images)
    rows = (n + cols - 1) // cols
    if figsize is None:
        figsize = (4.5 * cols, 4 * rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)
    for i in range(rows * cols):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        if i < n:
            ax.imshow(bgr_to_rgb(images[i]))
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=10)
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


def show_side_by_side(img1, img2, title1="", title2="", suptitle="", figsize=(12, 5)):
    """Show two images side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    ax1.imshow(bgr_to_rgb(img1))
    ax1.set_title(title1, fontsize=11)
    ax1.axis('off')
    ax2.imshow(bgr_to_rgb(img2))
    ax2.set_title(title2, fontsize=11)
    ax2.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


print("Display helpers ready.")

---
## Section 2 — Pipeline Overview Diagram

A visual map of the entire coverless steganography system.
The sender hides a message by selecting images from a shared database;
the receiver recovers the message by identifying those images after they
have been attacked (compressed, rotated, noised, etc.) during transmission.

In [ ]:
"""Draw the pipeline overview diagram with matplotlib.

Shows the full Sender → Channel → Receiver flow,
labelled with the section number where each stage is visualized below.
"""

from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0, 16)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_aspect('equal')

# Color scheme
sender_color = '#4ECDC4'
channel_color = '#FF6B6B'
receiver_color = '#45B7D1'
db_color = '#96CEB4'
text_color = '#2C3E50'

def draw_box(x, y, w, h, label, sublabel, color, section=None):
    """Draw a rounded box with label."""
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15",
                          facecolor=color, edgecolor=text_color, linewidth=1.5, alpha=0.85, zorder=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2 + 0.15, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color=text_color, zorder=3)
    ax.text(x + w/2, y + h/2 - 0.25, sublabel, ha='center', va='center',
            fontsize=7, color=text_color, alpha=0.8, zorder=3)
    if section:
        ax.text(x + w - 0.15, y + h - 0.15, section, ha='right', va='top',
                fontsize=7, color='white', fontweight='bold', zorder=3,
                bbox=dict(boxstyle='round,pad=0.15', facecolor='#E74C3C', edgecolor='none', alpha=0.9))

def draw_arrow(x1, y1, x2, y2, label=""):
    """Draw an arrow between boxes."""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=2, color=text_color, alpha=0.7))
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my + 0.2, label, ha='center', va='bottom', fontsize=7,
                color=text_color, fontstyle='italic')

# ─── SENDER SIDE ───
ax.text(3.5, 6.5, '📤 SENDER', fontsize=12, fontweight='bold', color=sender_color, ha='center')
draw_box(0.3, 4.8, 2.2, 1.2, 'Secret Message', 'e.g. "HELLO"', sender_color, '§4')
draw_box(3.0, 4.8, 2.2, 1.2, 'Segment into\nn-bit groups', '6-bit segments', sender_color, '§4')
draw_box(5.7, 4.8, 2.5, 1.2, 'Select Images\n(greedy search)', 'max segments/image', sender_color, '§4')

draw_arrow(2.5, 5.4, 3.0, 5.4, 'bytes→bits')
draw_arrow(5.2, 5.4, 5.7, 5.4, 'segments')

# ─── SHARED DATABASE ───
draw_box(5.7, 2.8, 2.5, 1.5, 'Shared Image\nDatabase', f'{len(ids)} VOC2012 images', db_color, '§3')
ax.annotate('', xy=(6.95, 4.8), xytext=(6.95, 4.3),
            arrowprops=dict(arrowstyle='->', lw=2, color=text_color, alpha=0.5))
ax.text(7.4, 4.55, 'lookup', fontsize=7, color=text_color, fontstyle='italic')

# ─── CHANNEL (attacks) ───
ax.text(10.5, 6.5, '⚡ CHANNEL', fontsize=12, fontweight='bold', color=channel_color, ha='center')
draw_box(9.0, 4.8, 3.0, 1.2, 'Attacks During\nTransmission', 'JPEG, rotation, noise...', channel_color, '§5')
draw_arrow(8.2, 5.4, 9.0, 5.4, 'stego images')

# ─── RECEIVER SIDE ───
ax.text(14.0, 6.5, '📥 RECEIVER', fontsize=12, fontweight='bold', color=receiver_color, ha='center')
draw_box(12.5, 4.8, 2.7, 1.2, 'Retrieve Original\n(SSCD search)', 'cosine similarity', receiver_color, '§6')
draw_arrow(12.0, 5.4, 12.5, 5.4, 'attacked imgs')

draw_box(12.5, 2.8, 2.7, 1.5, 'Decode Message\nfrom objects', 'labels→groups→bits', receiver_color, '§6')
ax.annotate('', xy=(13.85, 4.8), xytext=(13.85, 4.3),
            arrowprops=dict(arrowstyle='->', lw=2, color=text_color, alpha=0.5))
ax.text(14.3, 4.55, 'matched', fontsize=7, color=text_color, fontstyle='italic')

# Receiver also uses the shared DB
ax.annotate('', xy=(12.5, 3.55), xytext=(8.2, 3.55),
            arrowprops=dict(arrowstyle='->', lw=1.5, color=text_color, alpha=0.4, linestyle='dashed'))
ax.text(10.3, 3.75, 'same database (both sides have it)', fontsize=7,
        color=text_color, fontstyle='italic', ha='center')

# Output
draw_box(12.5, 1.0, 2.7, 1.2, 'Recovered\nMessage', '"HELLO" ✓', '#F9E79F', '§7')
ax.annotate('', xy=(13.85, 2.8), xytext=(13.85, 2.2),
            arrowprops=dict(arrowstyle='->', lw=2, color=text_color, alpha=0.5))

fig.suptitle('Coverless Steganography Pipeline', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

---
## Section 3 — Object Detection & Dictionary

The foundation of the system: Faster R-CNN detects objects in every image,
and a **dictionary** maps each object label to a binary group. When the
sender picks an image, its objects encode part of the secret message.

### 3a — Show sample images with detected objects

We draw bounding boxes on a few images from the database.
**Green** boxes = top-M objects (used for encoding), **gray** = below threshold.

In [ ]:
"""Pick sample images that have at least M objects above the confidence threshold,
so the visualization is interesting."""

# Find images with enough objects
sample_indices = []
for i, seq in enumerate(index.sequences):
    if len(seq) >= config.M:  # at least M objects
        sample_indices.append(i)
    if len(sample_indices) >= NUM_SAMPLE_IMAGES:
        break

sample_ids = [ids[i] for i in sample_indices]
print(f"Selected {len(sample_ids)} sample images: {sample_ids}")

In [ ]:
"""Draw bounding boxes on each sample image.

Green = top-M objects (above confidence threshold, sorted by area).
These are the objects that carry the secret message bits.
Gray dashed = other detected objects below the threshold.
"""

fig, axes = plt.subplots(2, 3, figsize=(16, 11))
axes = axes.flatten()

for idx, (sample_i, ax) in enumerate(zip(sample_indices, axes)):
    image = read_image(image_paths[ids[sample_i]])
    image_rgb = bgr_to_rgb(image.copy())
    all_objects = extraction.objects[sample_i]
    chosen = top_objects(all_objects, config.conf_threshold, config.M)
    chosen_labels = {(obj['label'], tuple(obj['box'])) for obj in chosen}

    ax.imshow(image_rgb)

    # Draw ALL detected objects (gray dashed = below threshold)
    for obj in all_objects:
        key = (obj['label'], tuple(obj['box']))
        x1, y1, x2, y2 = obj['box']
        is_chosen = key in chosen_labels
        color = '#2ECC71' if is_chosen else '#BDC3C7'
        lw = 2.5 if is_chosen else 1
        ls = '-' if is_chosen else '--'
        rect = Rectangle((x1, y1), x2 - x1, y2 - y1,
                         linewidth=lw, edgecolor=color, facecolor='none', linestyle=ls)
        ax.add_patch(rect)
        if is_chosen:
            # Show label and group assignment for top-M objects
            group = index.dictionary.get(obj['label'], '?')
            bits = segment_to_bits(group, config.n) if isinstance(group, int) else '?'
            label_text = f"{obj['label']} ({obj['score']:.0%})\nGroup {group} = {bits}"
            ax.text(x1, y1 - 3, label_text, fontsize=7, color='white',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='#27AE60', alpha=0.85))

    ax.set_title(f"{ids[sample_i]}\n{len(chosen)} top-M objects", fontsize=9)
    ax.axis('off')

# Hide unused axes
for j in range(len(sample_indices), len(axes)):
    axes[j].axis('off')

fig.suptitle(f'Object Detection: Top-{config.M} Objects per Image (threshold > {config.conf_threshold})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3b — The object→group dictionary

All detected labels are sorted alphabetically and split into **2^n groups**
using dynamic programming (minimises the imbalance of images per group).
Each group corresponds to an n-bit binary sequence.

In [ ]:
"""Display the object label → group → binary sequence dictionary.

This table is the core mapping that converts detected objects into bits.
Labels are sorted alphabetically; DP balances the number of images per group.
"""
import pandas as pd

dict_rows = []
for label in sorted(index.dictionary.keys()):
    group = index.dictionary[label]
    dict_rows.append({
        'Object Label': label,
        'Group': group,
        f'Binary ({config.n}-bit)': segment_to_bits(group, config.n),
        'Images with label': counts.get(label, 0),
    })

dict_df = pd.DataFrame(dict_rows)
print(f"Dictionary: {len(dict_df)} labels → {index.groups} groups (n={config.n}, 2^{config.n}={index.groups})")
print(f"\nFirst 20 entries:")
display(dict_df.head(20))
print(f"\n... ({len(dict_df)} labels total)")

In [ ]:
"""Visualize how many images each group covers (bar chart).

The DP split tries to make these bars as even as possible,
so each group has a similar number of images that can represent it.
"""

# Count images per group
group_image_counts = {}
for label, group in index.dictionary.items():
    group_image_counts[group] = group_image_counts.get(group, 0) + counts.get(label, 0)

groups_sorted = sorted(group_image_counts.keys())
fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(
    [segment_to_bits(g, config.n) for g in groups_sorted],
    [group_image_counts[g] for g in groups_sorted],
    color='#3498DB', alpha=0.8
)
ax.set_xlabel(f'{config.n}-bit Group Sequence', fontsize=11)
ax.set_ylabel('Number of Images', fontsize=11)
ax.set_title('Images per Dictionary Group (DP-balanced)', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=90, labelsize=6)
plt.tight_layout()
plt.show()

---
## Section 4 — Message Hiding (Sender Side)

The sender converts a secret message into bits, splits it into n-bit segments,
and selects images from the database whose top-M objects match consecutive segments.

**Change `SECRET_MESSAGE` at the top** to try a different message.

### 4a — Message → bytes → bits → segments

In [ ]:
"""Convert the secret message to bytes, show its binary representation,
and split it into n-bit segments.

Each segment is a group number (0 to 2^n - 1) that maps to one object label.
"""

message_bytes = SECRET_MESSAGE.encode('utf-8')
segments, padding = bytes_to_segments(message_bytes, config.n)

# Display the conversion step-by-step
print("═" * 70)
print(f"  SECRET MESSAGE: \"{SECRET_MESSAGE}\"")
print("═" * 70)

# Step 1: Message as bytes
print(f"\n1. As bytes ({len(message_bytes)} bytes):")
byte_strs = [f"  '{chr(b)}' → {b:3d} (0x{b:02X})" for b in message_bytes]
for s in byte_strs:
    print(s)

# Step 2: As a full binary string
full_bits = ''.join(format(b, '08b') for b in message_bytes)
print(f"\n2. As binary ({len(full_bits)} bits):")
# Show 8-bit groups with character labels
for i, b in enumerate(message_bytes):
    byte_bits = format(b, '08b')
    print(f"   '{chr(b)}': {byte_bits}", end="  ")
print()
print(f"   Full: {full_bits}")
if padding > 0:
    print(f"   + {padding} padding zeros = {full_bits + '0' * padding}")

# Step 3: Split into n-bit segments
padded_bits = full_bits + '0' * padding
print(f"\n3. Split into {config.n}-bit segments ({len(segments)} segments):")
for i, seg in enumerate(segments):
    bits = padded_bits[i * config.n : (i + 1) * config.n]
    print(f"   Segment {i:2d}: {bits} = group {seg:2d}", end="")
    if i == len(segments) - 1 and padding > 0:
        print(f"  (last {padding} bits are padding)", end="")
    print()

print(f"\n   Total: {len(segments)} segments × {config.n} bits = {len(segments) * config.n} bits")
print(f"   Padding: {padding} bits")
print(f"   Effective: {len(message_bytes) * 8} message bits")

### 4b — Segments → binary sequence visualization

In [ ]:
"""Visualize the segment sequence as a coloured strip.

Each cell shows the group number and its binary representation.
This strip IS the secret — the sender must find images whose objects match it.
"""

n_segs = len(segments)
fig, ax = plt.subplots(figsize=(max(14, n_segs * 1.2), 2))
cmap = plt.cm.Set3

for i, seg in enumerate(segments):
    color = cmap(seg / max(index.groups - 1, 1))
    rect = plt.Rectangle((i, 0), 0.9, 1, facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(rect)
    bits_str = segment_to_bits(seg, config.n)
    ax.text(i + 0.45, 0.65, f'{seg}', ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(i + 0.45, 0.3, bits_str, ha='center', va='center', fontsize=7, family='monospace')

ax.set_xlim(-0.2, n_segs + 0.2)
ax.set_ylim(-0.1, 1.3)
ax.set_title(f'Secret Segment Sequence for "{SECRET_MESSAGE}"', fontsize=13, fontweight='bold')
ax.set_xlabel('Segment position', fontsize=10)
ax.set_xticks(np.arange(n_segs) + 0.45)
ax.set_xticklabels(range(n_segs), fontsize=8)
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.show()

### 4c — Image selection (greedy search)

The sender picks images from the database whose top-M objects match as many
consecutive segments as possible. This is the **greedy hiding algorithm** (§III-B).

In [ ]:
"""Run the hiding algorithm and show which images were selected.

Each stego image carries 1 to M segments of the secret message.
The auxiliary info tuple tells the receiver which objects carry
which segments and in what order.
"""

rng = np.random.default_rng(config.seed)
result = hide_segments(segments, index, rng, padding)

print(f"Message: \"{SECRET_MESSAGE}\" ({len(segments)} segments)")
print(f"Complete: {result.complete}")
print(f"Images selected: {len(result.stego)}")
print(f"Segments hidden: {result.segments_hidden}/{result.segments_total}")
print(f"Repeated images: {result.repeated_images}")
print()

# Show each selected image and its role
seg_pos = 0
for img_num, item in enumerate(result.stego):
    img_id = ids[item.image_index]
    n_mapped = item.aux.mapped_count
    seg_range = segments[seg_pos : seg_pos + n_mapped]
    seg_bits = [segment_to_bits(s, config.n) for s in seg_range]

    # Get the object labels for this image
    chosen = top_objects(extraction.objects[item.image_index], config.conf_threshold, config.M)
    labels = [obj['label'] for obj in chosen]

    print(f"  Image {img_num + 1}: {img_id}")
    print(f"    Objects: {labels}")
    print(f"    Carries segments {seg_pos}–{seg_pos + n_mapped - 1}: groups {seg_range} = {seg_bits}")
    print(f"    Aux order: {item.aux.order}, cluster: C{item.aux.cluster}")
    seg_pos += n_mapped

### 4d — Visualize the stego images with object annotations

In [ ]:
"""Show each stego image with its objects annotated.

Objects used for the message are highlighted in green with their
group assignment. Objects not used are shown in gray.
This is what the sender transmits: an ordered sequence of ordinary images.
"""

n_stego = len(result.stego)
cols = min(n_stego, 4)
rows = (n_stego + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if n_stego == 1:
    axes = np.array([[axes]])
axes = np.atleast_2d(axes)

seg_pos = 0
for img_num, item in enumerate(result.stego):
    r, c = divmod(img_num, cols)
    ax = axes[r, c]
    image = read_image(image_paths[ids[item.image_index]])
    ax.imshow(bgr_to_rgb(image))

    chosen = top_objects(extraction.objects[item.image_index], config.conf_threshold, config.M)
    n_mapped = item.aux.mapped_count

    for obj_idx, obj in enumerate(chosen):
        x1, y1, x2, y2 = obj['box']
        position_in_message = item.aux.order[obj_idx] if obj_idx < len(item.aux.order) else 0

        if position_in_message > 0:
            # This object carries a message segment
            seg_idx = seg_pos + position_in_message - 1
            group = segments[seg_idx] if seg_idx < len(segments) else '?'
            bits = segment_to_bits(group, config.n) if isinstance(group, int) else '?'
            rect = Rectangle((x1, y1), x2 - x1, y2 - y1,
                             lw=2.5, edgecolor='#2ECC71', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 3, f"#{position_in_message}: {obj['label']}\n→ {bits}",
                    fontsize=7, color='white',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='#27AE60', alpha=0.9))
        else:
            rect = Rectangle((x1, y1), x2 - x1, y2 - y1,
                             lw=1, edgecolor='#BDC3C7', facecolor='none', linestyle='--')
            ax.add_patch(rect)

    ax.set_title(f'Stego Image {img_num + 1}: {ids[item.image_index]}\n'
                 f'Carries {n_mapped} segment(s)', fontsize=9)
    ax.axis('off')
    seg_pos += n_mapped

# Hide unused axes
for i in range(n_stego, rows * cols):
    r, c = divmod(i, cols)
    axes[r, c].axis('off')

fig.suptitle(f'Stego Image Sequence for "{SECRET_MESSAGE}" — This is what the sender transmits',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 5 — Attack Gallery

During transmission, the stego images may be **attacked**: compressed,
rotated, cropped, noised, or filtered. The system must recover the
original image despite these distortions.

We show **all 34 attacks** (22 original + 12 extended) applied to one
stego image, with the **SSCD cosine similarity** score for each.

### 5a — Pick a stego image and build attacks

In [ ]:
"""Select the first stego image as the attack demo target.
Also build all 34 attacks (22 original from Table IX + 12 extended).
"""

demo_item = result.stego[0]
demo_id = ids[demo_item.image_index]
demo_image = read_image(image_paths[demo_id])
demo_descriptor = descriptors[demo_item.image_index]

# Build all attacks (including extended)
all_attacks = build_attacks(config.gamma_exponent, extended=True)
attack_names_no_ctrl = list(TABLE_IX_ORDER) + list(EXTENDED_ORDER)

print(f"Demo image: {demo_id}")
print(f"Image shape: {demo_image.shape}")
print(f"Total attacks: {len(attack_names_no_ctrl)} (22 original + 12 extended)")

### 5b — Apply all attacks and compute SSCD similarity

In [ ]:
"""Apply every attack to the demo image and compute the SSCD cosine similarity
between the original and attacked versions.

Similarity close to 1.0 means SSCD recognizes the attacked image as the same
image — retrieval succeeds. This is why SSCD gets 100% on VOC2012.
"""

attacked_images = {}
sscd_scores = {}

for name in attack_names_no_ctrl:
    rng_attack = attack_rng(config.seed, name, 0, 0)
    attacked = all_attacks[name](demo_image, rng_attack)
    attacked_images[name] = attacked

    # Compute SSCD similarity
    attacked_desc = sscd_model.describe([attacked])[0]
    similarity = float(demo_descriptor @ attacked_desc)
    sscd_scores[name] = similarity

print(f"All {len(attacked_images)} attacks applied.")
print(f"\nSSCD similarity scores:")
for name in attack_names_no_ctrl:
    score = sscd_scores[name]
    status = '✓' if score > 0.5 else '✗'
    print(f"  {status} {name:40s} {score:.4f}")

### 5c — Attack gallery: geometric attacks (13)

In [ ]:
"""Show all 13 geometric attacks in a grid.

Geometric attacks change the shape/position of the image:
cropping, rotation, translation, scaling.
The SSCD score is shown below each image.
"""

geo_names = list(GEOMETRIC)
n_geo = len(geo_names)
cols = 4
rows = (n_geo + 1 + cols - 1) // cols  # +1 for the original

fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
axes = axes.flatten()

# First panel: original
axes[0].imshow(bgr_to_rgb(demo_image))
axes[0].set_title('ORIGINAL', fontsize=10, fontweight='bold', color='#2C3E50')
axes[0].axis('off')

for i, name in enumerate(geo_names):
    ax = axes[i + 1]
    ax.imshow(bgr_to_rgb(attacked_images[name]))
    score = sscd_scores[name]
    color = '#27AE60' if score > 0.5 else '#E74C3C'
    ax.set_title(f'{name}\nSSCD: {score:.4f}', fontsize=8, color=color, fontweight='bold')
    ax.axis('off')

for j in range(n_geo + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Geometric Attacks (13)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5d — Attack gallery: non-geometric attacks (9)

In [ ]:
"""Show all 9 non-geometric attacks.

Non-geometric attacks change pixel values without moving them:
noise, filters, JPEG compression, histogram/gamma adjustments.
"""

nongeo_names = list(NON_GEOMETRIC)
n_ng = len(nongeo_names)
cols = 4
rows = (n_ng + 1 + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
axes = axes.flatten()

axes[0].imshow(bgr_to_rgb(demo_image))
axes[0].set_title('ORIGINAL', fontsize=10, fontweight='bold', color='#2C3E50')
axes[0].axis('off')

for i, name in enumerate(nongeo_names):
    ax = axes[i + 1]
    ax.imshow(bgr_to_rgb(attacked_images[name]))
    score = sscd_scores[name]
    color = '#27AE60' if score > 0.5 else '#E74C3C'
    ax.set_title(f'{name}\nSSCD: {score:.4f}', fontsize=8, color=color, fontweight='bold')
    ax.axis('off')

for j in range(n_ng + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Non-Geometric Attacks (9)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5e — Attack gallery: extended attacks (12)

Combined attacks chain multiple operations (e.g. JPEG + rotation + noise).
Heavy attacks use stronger parameters than the originals.

In [ ]:
"""Show the 12 extended attacks: 6 combined + 6 heavy.

These go beyond the paper's original 22 attacks to stress-test retrieval.
They are NOT used in the retrievability pre-check, so they are genuinely
unseen during filtering.
"""

ext_names = list(EXTENDED_ORDER)
n_ext = len(ext_names)
cols = 4
rows = (n_ext + 1 + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
axes = axes.flatten()

axes[0].imshow(bgr_to_rgb(demo_image))
axes[0].set_title('ORIGINAL', fontsize=10, fontweight='bold', color='#2C3E50')
axes[0].axis('off')

for i, name in enumerate(ext_names):
    ax = axes[i + 1]
    ax.imshow(bgr_to_rgb(attacked_images[name]))
    score = sscd_scores[name]
    color = '#27AE60' if score > 0.5 else '#E74C3C'
    # Indicate combined vs heavy
    tag = '[COMBINED]' if name in COMBINED else '[HEAVY]'
    ax.set_title(f'{tag} {name}\nSSCD: {score:.4f}', fontsize=7, color=color, fontweight='bold')
    ax.axis('off')

for j in range(n_ext + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Extended Attacks: Combined (6) + Heavy (6)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5f — SSCD similarity score chart across all attacks

In [ ]:
"""Bar chart of SSCD cosine similarity for all 34 attacks.

Every bar above the dashed threshold line means the attack is defeated:
SSCD recognizes the original image despite the attack.
"""

fig, ax = plt.subplots(figsize=(16, 5))

all_names = list(TABLE_IX_ORDER) + list(EXTENDED_ORDER)
scores = [sscd_scores[n] for n in all_names]
colors = []
for name in all_names:
    if name in GEOMETRIC:
        colors.append('#3498DB')  # blue for geometric
    elif name in NON_GEOMETRIC:
        colors.append('#2ECC71')  # green for non-geometric
    elif name in COMBINED:
        colors.append('#E67E22')  # orange for combined
    else:
        colors.append('#E74C3C')  # red for heavy

bars = ax.bar(range(len(all_names)), scores, color=colors, alpha=0.85)
ax.axhline(y=0.5, color='#E74C3C', linestyle='--', linewidth=1.5, alpha=0.7, label='Threshold (0.5)')
ax.set_xticks(range(len(all_names)))
ax.set_xticklabels(all_names, rotation=90, fontsize=6)
ax.set_ylabel('SSCD Cosine Similarity', fontsize=11)
ax.set_title(f'SSCD Robustness: All 34 Attacks on "{demo_id}"', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498DB', alpha=0.85, label='Geometric (13)'),
    Patch(facecolor='#2ECC71', alpha=0.85, label='Non-geometric (9)'),
    Patch(facecolor='#E67E22', alpha=0.85, label='Combined (6)'),
    Patch(facecolor='#E74C3C', alpha=0.85, label='Heavy (6)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

---
## Section 6 — Retrieval & Decoding (Receiver Side)

The receiver gets the (attacked) stego images. For each one:
1. Compute its SSCD descriptor
2. Search the entire database by cosine similarity
3. Identify the original image (rank-1 match)
4. Read its objects → groups → bits
5. Reassemble the message

### 6a — Build the SSCD retriever

In [ ]:
"""Create the Phase 3 SSCD retriever.

This is the receiver's tool: given an attacked image, it searches
the entire database (all 11540 images) to find the original.
"""

# Build the SSCD retriever (searches all images, no cluster needed)
retriever = DescriptorRetriever(
    index=index,
    paths=image_paths,
    descriptors=descriptors,
    model=sscd_model,
    mode='sscd',
    top_k=config.retrieval_top_k,
    size=config.ssim_size,
)
print(f"SSCD retriever ready.")
print(f"Searching over {len(retriever.search)} images.")
print(f"Descriptor dimension: {sscd_model.dim}")

### 6b — Detailed retrieval walkthrough for one attack

Walk through the receiver's process step-by-step for a single attacked stego image.

In [ ]:
"""Walk through the receiver's process for one attacked stego image.

1. Receive the attacked image
2. Compute its SSCD descriptor
3. Search the database — show top-5 matches
4. The best match is the original stego image
5. Decode its objects into the secret bits
"""

# Attack the first stego image
demo_stego = result.stego[0]
original_img = read_image(image_paths[ids[demo_stego.image_index]])
rng_demo = attack_rng(config.seed, DEMO_ATTACK, 0, 0)
attacked_img = all_attacks[DEMO_ATTACK](original_img, rng_demo)

print(f"Attack: {DEMO_ATTACK}")
print(f"Original: {ids[demo_stego.image_index]}")

# Show original vs attacked
show_side_by_side(
    original_img, attacked_img,
    'Original stego image', f'After: {DEMO_ATTACK}',
    suptitle='Step 1: Receiver gets the attacked image'
)

In [ ]:
"""Step 2-3: Compute SSCD descriptor and search the database.

Show the top-5 matches ranked by cosine similarity.
The correct original should be rank #1.
"""

# Compute similarity of attacked image against entire database
cosine_scores = retriever.similarities(attacked_img)
top_k_indices = np.argsort(-cosine_scores)[:5]

# Display top-5 matches
fig, axes = plt.subplots(1, 6, figsize=(18, 4))

# Query image
axes[0].imshow(bgr_to_rgb(attacked_img))
axes[0].set_title(f'QUERY\n({DEMO_ATTACK})', fontsize=9, fontweight='bold', color='#E74C3C')
axes[0].axis('off')

# Top-5 results
for rank, idx in enumerate(top_k_indices):
    ax = axes[rank + 1]
    actual_image_idx = int(retriever.search[idx])
    match_img = read_image(image_paths[ids[actual_image_idx]])
    ax.imshow(bgr_to_rgb(match_img))
    score = float(cosine_scores[idx])
    is_correct = actual_image_idx == demo_stego.image_index
    border_color = '#27AE60' if is_correct else '#BDC3C7'
    title_color = '#27AE60' if is_correct else '#2C3E50'
    check = ' ✓' if is_correct else ''
    ax.set_title(f'Rank #{rank + 1}{check}\n{ids[actual_image_idx]}\ncos={score:.4f}',
                 fontsize=8, color=title_color, fontweight='bold' if is_correct else 'normal')
    for spine in ax.spines.values():
        spine.set_color(border_color)
        spine.set_linewidth(3 if is_correct else 1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle('Step 2-3: SSCD Search — Top 5 Matches from 11,540 Images',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 6c — Decode the message from the retrieved image

In [ ]:
"""Step 4-5: Decode the secret message from the matched image's objects.

The receiver knows the auxiliary info (object order, cluster, padding)
and the dictionary, so it can read the objects → groups → bits → bytes.

We do this for ALL stego images, not just the first one, to recover
the complete message.
"""

print("═" * 70)
print("  RECEIVER: DECODING THE COMPLETE MESSAGE")
print("═" * 70)

all_decoded_segments = []

for img_num, item in enumerate(result.stego):
    original = read_image(image_paths[ids[item.image_index]])
    rng_r = attack_rng(config.seed, DEMO_ATTACK, 0, img_num)
    attacked = all_attacks[DEMO_ATTACK](original, rng_r)

    # Receiver retrieves the original
    found = retriever.retrieve(attacked)
    is_correct = found.image_index == item.image_index

    # Decode segments
    decoded = decode_image(found.image_index, item.aux, index)
    all_decoded_segments.extend(decoded)

    print(f"\n  Image {img_num + 1}: {ids[item.image_index]}")
    print(f"    Retrieved: {ids[found.image_index]} {'✓ CORRECT' if is_correct else '✗ WRONG'}")
    print(f"    Similarity: {found.best_score:.4f} (2nd best: {found.second_score:.4f})")
    print(f"    Decoded segments: {decoded}")
    print(f"    As bits: {[segment_to_bits(s, config.n) if s is not None else '?' for s in decoded]}")

# Reassemble the message
recovered_bytes = segments_to_bytes(all_decoded_segments, config.n, padding)
try:
    recovered_text = recovered_bytes.decode('utf-8')
except UnicodeDecodeError:
    recovered_text = repr(recovered_bytes)

match = recovered_bytes == message_bytes

print("\n" + "═" * 70)
print(f"  RECOVERED MESSAGE: \"{recovered_text}\"")
print(f"  ORIGINAL MESSAGE:  \"{SECRET_MESSAGE}\"")
print(f"  MATCH: {'✓ PERFECT' if match else '✗ MISMATCH'}")
print("═" * 70)

### 6d — Test retrieval across multiple attack types

In [ ]:
"""Run the full encode→attack→retrieve→decode cycle for several attacks.

This proves the message survives diverse attack types.
We test a sample of attacks from each category.
"""

test_attacks = [
    'Rotation 30°',
    'Center Cropping 20%',
    'JPEG Compression',
    'Gauss Noise',
    'Scaling',
    'JPEG Q50 + Rotation 15°',
    'JPEG Quality 10',
    'Heavy Salt & Pepper',
]

print(f"Testing message recovery for \"{SECRET_MESSAGE}\" across {len(test_attacks)} attacks:\n")

results_table = []
for atk_name in test_attacks:
    decoded_all = []
    all_correct = True
    for img_num, item in enumerate(result.stego):
        original = read_image(image_paths[ids[item.image_index]])
        rng_t = attack_rng(config.seed, atk_name, 0, img_num)
        attacked = all_attacks[atk_name](original, rng_t)
        found = retriever.retrieve(attacked)
        if found.image_index != item.image_index:
            all_correct = False
        decoded_all.extend(decode_image(found.image_index, item.aux, index))

    recovered = segments_to_bytes(decoded_all, config.n, padding)
    match = recovered == message_bytes
    try:
        text = recovered.decode('utf-8')
    except:
        text = repr(recovered)

    status = '✓' if match else '✗'
    results_table.append({
        'Attack': atk_name,
        'Retrieval': '✓ All correct' if all_correct else '✗ Mismatch',
        'Recovered': text,
        'Match': status
    })
    print(f"  {status} {atk_name:40s} → \"{text}\"")

print(f"\nAll attacks defeated: {all(r['Match'] == '✓' for r in results_table)}")

---
## Section 7 — Interactive Upload Demo

**You are the receiver!**

Upload an attacked photo and the system will:
1. Compute its SSCD descriptor
2. Search the database for the best match
3. Show the matched image and its objects
4. Decode whatever message those objects carry

To test it: save one of the attacked images from Section 5 and upload it.

### 7a — Save some attacked images for uploading

This cell saves attacked versions of the stego images so you can download
and re-upload them to test the demo. You can also use any image from the
VOC2012 dataset.

In [ ]:
"""Save a few attacked stego images to disk so the user can download
and re-upload them in the demo cell below.
"""

save_dir = Path("/kaggle/working/demo_attacked")
save_dir.mkdir(exist_ok=True)

saved_files = []
demo_attacks_to_save = ['Rotation 30°', 'JPEG Compression', 'Heavy Gauss Noise',
                         'JPEG Q50 + Rotation 15°', 'JPEG Quality 10']

for atk_name in demo_attacks_to_save:
    rng_s = attack_rng(config.seed, atk_name, 0, 0)
    attacked = all_attacks[atk_name](demo_image, rng_s)
    safe_name = atk_name.replace(' ', '_').replace('°', 'deg').replace('+', 'and')
    safe_name = safe_name.replace('×', 'x').replace('/', '_')
    filename = f"attacked_{demo_id}_{safe_name}.jpg"
    filepath = save_dir / filename
    cv2.imwrite(str(filepath), attacked)
    saved_files.append(filename)
    print(f"Saved: {filename}")

print(f"\nDownload from: {save_dir}")
print("Then upload in the next cell to test the receiver.")

### 7b — Upload and decode

Upload any image. The receiver searches the database for the closest match
and decodes whatever message the matched image's objects encode.

In [ ]:
"""Interactive upload demo.

Upload an attacked image (or any image). The receiver:
1. Computes its SSCD descriptor
2. Searches all 11,540 database images
3. Shows the top match and its similarity score
4. Reads the matched image's objects and decodes the group sequence

On Kaggle, you can use the file upload widget or type a path to any image.
If neither is available, it automatically uses a saved attacked image.
"""

def decode_uploaded_image(image_bgr, label="uploaded"):
    """Run the receiver pipeline on a single image."""
    # Step 1: Compute SSCD descriptor
    uploaded_desc = sscd_model.describe([image_bgr])[0]

    # Step 2: Search the database
    all_scores = retriever.similarities(image_bgr)
    best_idx = int(np.argmax(all_scores))
    best_score = float(all_scores[best_idx])
    best_image_idx = int(retriever.search[best_idx])
    best_id = ids[best_image_idx]

    # Top-5
    top5 = np.argsort(-all_scores)[:5]
    print(f"  Top 5 matches:")
    for rank, ti in enumerate(top5):
        ii = int(retriever.search[ti])
        print(f"    #{rank+1}: {ids[ii]:20s}  cos={float(all_scores[ti]):.4f}")

    # Step 3: Show side by side
    matched_img = read_image(image_paths[best_id])
    show_side_by_side(
        image_bgr, matched_img,
        f'Uploaded: {label}',
        f'Best match: {best_id} (cos={best_score:.4f})',
        suptitle='Receiver: Uploaded vs. Database Match'
    )

    # Step 4: Decode the matched image's objects
    matched_objects = top_objects(extraction.objects[best_image_idx], config.conf_threshold, config.M)
    matched_labels = [obj['label'] for obj in matched_objects]
    matched_groups = [index.dictionary.get(label, -1) for label in matched_labels]
    matched_bits = [segment_to_bits(g, config.n) if g >= 0 else '?' for g in matched_groups]

    print(f"\n  Matched image: {best_id}")
    print(f"  Objects detected: {matched_labels}")
    print(f"  Group sequence:   {matched_groups}")
    print(f"  Binary sequence:  {matched_bits}")

    # Check if this is one of our stego images
    is_stego = False
    for s_num, s_item in enumerate(result.stego):
        if s_item.image_index == best_image_idx:
            is_stego = True
            decoded_segs = decode_image(s_item.image_index, s_item.aux, index)
            decoded_bits = [segment_to_bits(s, config.n) if s is not None else '?' for s in decoded_segs]
            print(f"\n  ✓ This is stego image #{s_num + 1} from our message!")
            print(f"    Message segments: {decoded_segs}")
            print(f"    Message bits:     {decoded_bits}")
            break

    if not is_stego:
        print(f"\n  ℹ This image is not one of the stego images for \"{SECRET_MESSAGE}\".")
        print(f"    It still has objects that encode group sequences, but they are not part of our message.")


# ─── Try multiple upload methods ───

uploaded_something = False

# Method 1: Try ipywidgets FileUpload (works on Kaggle with interaction enabled)
try:
    import ipywidgets as widgets
    from IPython.display import display as ipy_display

    uploader = widgets.FileUpload(accept='image/*', multiple=False,
                                   description='Upload Image')
    out = widgets.Output()

    def on_upload(change):
        with out:
            out.clear_output()
            for name, file_info in uploader.value.items():
                content = file_info['content']
                arr = np.frombuffer(content, np.uint8)
                img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
                if img is None:
                    print(f"ERROR: Could not read {name} as an image.")
                    continue
                print(f"\nProcessing: {name}")
                decode_uploaded_image(img, name)

    uploader.observe(on_upload, names='value')
    print("Upload an image using the button below:")
    print("(Tip: download one of the attacked images from Section 7a)")
    ipy_display(uploader)
    ipy_display(out)
    uploaded_something = True

except Exception:
    pass

# Method 2: Fallback — use the saved attacked images from Section 7a
if not uploaded_something:
    print("File upload widget not available.")
    print("Using saved attacked images from Section 7a instead.\n")

    print("Available attacked images:")
    for i, fname in enumerate(saved_files):
        print(f"  [{i}] {fname}")

    # Decode the first saved image as a demo
    fallback_path = save_dir / saved_files[0]
    fallback_img = cv2.imread(str(fallback_path), cv2.IMREAD_COLOR)
    print(f"\nProcessing: {saved_files[0]}")
    decode_uploaded_image(fallback_img, saved_files[0])

    print(f"\n{'─' * 60}")
    print("To test with a different image, change the index below and re-run:")
    print("  fallback_path = save_dir / saved_files[INDEX]")
    print("  fallback_img = cv2.imread(str(fallback_path), cv2.IMREAD_COLOR)")
    print("  decode_uploaded_image(fallback_img, saved_files[INDEX])")
    print(f"\nOr provide any image path:")
    print("  img = cv2.imread('/kaggle/input/path/to/image.jpg', cv2.IMREAD_COLOR)")
    print("  decode_uploaded_image(img, 'my_image.jpg')")

---
## Section 8 — Phase 2 vs Phase 3 Results Comparison

This section compares the **original paper's retrieval** (Phase 2: cluster + SSIM)
with the **enhanced retrieval** (Phase 3: SSCD whole-database search).

We load the experiment results from completed runs and build comparison charts.

### 8a — Load experiment results

In [ ]:
"""Load the experiment results from saved runs.

We look for runs with different retrieval modes:
- cluster_ssim: the paper's Phase 2 baseline
- sscd: Phase 3 with SSCD cosine search
- sscd_align: Phase 3 with SSCD + Fourier-Mellin alignment
- sscd_ext / sscd_align_ext: extended attack variants
"""
import pandas as pd

runs_dir = Path("experiments/runs")
run_data = {}  # tag → {summary, attacks_df}

if runs_dir.is_dir():
    for run_folder in sorted(runs_dir.iterdir()):
        summary_path = run_folder / "summary.json"
        attacks_path = run_folder / "attacks.csv"
        if summary_path.is_file() and attacks_path.is_file():
            summary = json.loads(summary_path.read_text())
            attacks_df = pd.read_csv(attacks_path)
            tag = summary['config'].get('retrieval', 'cluster_ssim')
            if summary['config'].get('extended_attacks', False):
                tag += '_ext'
            if summary['config'].get('usable_threshold', 0) > 0:
                tag += f"_usable{summary['config']['usable_threshold']:g}"
            run_data[tag] = {
                'summary': summary,
                'attacks': attacks_df,
                'folder': run_folder.name,
            }
            print(f"Loaded: {run_folder.name} → tag: {tag}")
            print(f"  Retrieval: {summary['config']['retrieval']}, "
                  f"Extended: {summary['config'].get('extended_attacks', False)}, "
                  f"Attacks in CSV: {len(attacks_df['attack'].unique())}")

if not run_data:
    print("\nNo experiment runs found. Make sure the Phase 2 and Phase 3 notebook outputs")
    print("are attached as input datasets. The comparison charts below will be skipped.")
else:
    print(f"\nTotal runs loaded: {len(run_data)}")

### 8b — Per-attack accuracy comparison (22 original attacks)

In [ ]:
"""Bar chart comparing per-attack accuracy for cluster_ssim vs sscd vs sscd_align.

This is the key chart: it shows cluster_ssim failing on geometric attacks
while SSCD stays at 100% across the board.
"""

# Modes to compare (use non-extended versions for the 22-attack comparison)
compare_modes = ['cluster_ssim', 'sscd', 'sscd_align']
mode_labels = {
    'cluster_ssim': 'Phase 2: Cluster+SSIM',
    'sscd': 'Phase 3: SSCD',
    'sscd_align': 'Phase 3: SSCD+Align',
}
mode_colors = {
    'cluster_ssim': '#E74C3C',
    'sscd': '#3498DB',
    'sscd_align': '#2ECC71',
}

# Collect accuracy per attack for each mode
available_modes = [m for m in compare_modes if m in run_data]
if not available_modes:
    print("No matching runs found for comparison. Skipping.")
else:
    attack_order = list(TABLE_IX_ORDER)
    accuracy_by_mode = {}

    for mode in available_modes:
        df = run_data[mode]['attacks']
        acc_per_attack = {}
        for attack_name in attack_order:
            rows = df[df['attack'] == attack_name]
            if not rows.empty:
                acc_per_attack[attack_name] = rows['acc_percent'].iloc[0]
        accuracy_by_mode[mode] = acc_per_attack

    # Plot
    n_attacks = len(attack_order)
    n_modes = len(available_modes)
    width = 0.8 / n_modes
    x = np.arange(n_attacks)

    fig, ax = plt.subplots(figsize=(18, 6))

    for i, mode in enumerate(available_modes):
        values = [accuracy_by_mode[mode].get(a, 0) for a in attack_order]
        offset = (i - (n_modes - 1) / 2) * width
        ax.bar(x + offset, values, width * 0.9,
               label=mode_labels.get(mode, mode),
               color=mode_colors.get(mode, f'C{i}'), alpha=0.85)

    ax.axhline(y=100, color='#2ECC71', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(attack_order, rotation=70, ha='right', fontsize=7)
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title('Per-Attack Accuracy: Phase 2 (Cluster+SSIM) vs Phase 3 (SSCD)',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, loc='lower left')
    ax.set_ylim(0, 105)

    # Shade geometric vs non-geometric regions
    ax.axvspan(-0.5, len(GEOMETRIC) - 0.5, alpha=0.05, color='blue', label='_')
    ax.axvspan(len(GEOMETRIC) - 0.5, n_attacks - 0.5, alpha=0.05, color='green', label='_')
    ax.text(len(GEOMETRIC) / 2, 103, 'GEOMETRIC', ha='center', fontsize=8, color='#3498DB', alpha=0.7)
    ax.text(len(GEOMETRIC) + len(NON_GEOMETRIC) / 2, 103, 'NON-GEOMETRIC',
            ha='center', fontsize=8, color='#27AE60', alpha=0.7)

    plt.tight_layout()
    plt.show()

### 8c — Why Cluster+SSIM fails: visual proof

Pick an attack where cluster_ssim fails and show the SSIM vs SSCD scores.
This makes the accuracy numbers concrete.

In [ ]:
"""Visual proof of why SSIM fails on geometric attacks.

Compare SSIM and SSCD scores for the same image under rotation.
SSIM compares pixels position-by-position, so rotation destroys the score.
SSCD compares learned representations that are rotation-invariant.
"""
from stego.recovery import to_comparison_gray, equation_2

# Pick a sample image
sample_idx = sample_indices[0]
sample_img = read_image(image_paths[ids[sample_idx]])
sample_desc = descriptors[sample_idx]

# Test attacks that show the difference
test_names = ['Rotation 10°', 'Rotation 30°', 'Rotation 50°',
              'JPEG Compression', 'Gauss Noise', 'Median Filter']

fig, axes = plt.subplots(2, len(test_names) + 1, figsize=(20, 8))

# Row 1: SSIM scores
# Row 2: SSCD scores
gray_orig = to_comparison_gray(sample_img, config.ssim_size)

# Original
for row in range(2):
    axes[row, 0].imshow(bgr_to_rgb(sample_img))
    metric = 'SSIM' if row == 0 else 'SSCD'
    axes[row, 0].set_title(f'Original\n{metric}: 1.0000', fontsize=9, fontweight='bold')
    axes[row, 0].axis('off')

axes[0, 0].set_ylabel('SSIM Score\n(pixel comparison)', fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel('SSCD Score\n(learned features)', fontsize=10, fontweight='bold')

for col, atk_name in enumerate(test_names, 1):
    rng_v = attack_rng(config.seed, atk_name, 0, 0)
    attacked = all_attacks[atk_name](sample_img, rng_v)

    # SSIM score
    gray_atk = to_comparison_gray(attacked, config.ssim_size)
    ssim_score = equation_2(gray_orig, gray_atk)

    # SSCD score
    atk_desc = sscd_model.describe([attacked])[0]
    sscd_score = float(sample_desc @ atk_desc)

    for row, (score, label) in enumerate([(ssim_score, 'SSIM'), (sscd_score, 'SSCD')]):
        axes[row, col].imshow(bgr_to_rgb(attacked))
        color = '#27AE60' if score > 0.5 else '#E74C3C'
        axes[row, col].set_title(f'{atk_name}\n{label}: {score:.4f}',
                                 fontsize=8, color=color, fontweight='bold')
        axes[row, col].axis('off')

fig.suptitle('Why Cluster+SSIM Fails: SSIM vs SSCD Scores Under Attack',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 8d — Attack category breakdown (heatmap)

In [ ]:
"""Heatmap showing mean accuracy per attack category per retrieval mode.

Categories: Geometric, Non-geometric, Combined (extended), Heavy (extended).
This highlights that geometric attacks are Phase 2's weakness.
"""

if not run_data:
    print("No runs loaded. Skipping.")
else:
    categories = {
        'Geometric (13)': list(GEOMETRIC),
        'Non-Geometric (9)': list(NON_GEOMETRIC),
    }
    # Add extended categories if we have ext runs
    ext_modes = [m for m in run_data if '_ext' in m]
    if ext_modes:
        categories['Combined (6)'] = list(COMBINED)
        categories['Heavy (6)'] = list(HEAVY)

    # Compute mean accuracy per category per mode
    all_modes = sorted(run_data.keys())
    heatmap_data = []
    mode_display_names = []

    for mode in all_modes:
        df = run_data[mode]['attacks']
        row = []
        for cat_name, cat_attacks in categories.items():
            present = df[df['attack'].isin(cat_attacks)]
            mean_acc = present['acc_percent'].mean() if not present.empty else float('nan')
            row.append(mean_acc)
        heatmap_data.append(row)
        display_name = mode_labels.get(mode, mode)
        mode_display_names.append(display_name)

    heatmap_array = np.array(heatmap_data)

    fig, ax = plt.subplots(figsize=(10, max(3, len(all_modes) * 0.8 + 1)))
    im = ax.imshow(heatmap_array, cmap='RdYlGn', vmin=50, vmax=100, aspect='auto')
    ax.set_xticks(range(len(categories)))
    ax.set_xticklabels(categories.keys(), fontsize=10)
    ax.set_yticks(range(len(mode_display_names)))
    ax.set_yticklabels(mode_display_names, fontsize=10)

    # Annotate cells
    for i in range(len(mode_display_names)):
        for j in range(len(categories)):
            val = heatmap_array[i, j]
            if not np.isnan(val):
                text_color = 'white' if val < 70 else 'black'
                ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                        fontsize=11, fontweight='bold', color=text_color)
            else:
                ax.text(j, i, 'N/A', ha='center', va='center',
                        fontsize=10, color='gray')

    plt.colorbar(im, ax=ax, label='Accuracy (%)', shrink=0.8)
    ax.set_title('Mean Accuracy by Attack Category and Retrieval Mode',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 8e — Capacity comparison

Phase 3 doesn't send a cluster ID (it searches all images), so the
auxiliary cost per image is lower. Also compare images needed per message.

In [ ]:
"""Compare capacity metrics across retrieval modes.

Key metrics:
- Images needed to hide messages of different sizes
- Bits per image (how much data one image carries)
- Auxiliary bits (overhead: cluster ID, object ordering)
- Search space size (cluster members vs whole database)
"""

if not run_data:
    print("No runs loaded. Skipping.")
else:
    print("Capacity Comparison")
    print("=" * 80)

    capacity_rows = []
    for mode in sorted(run_data.keys()):
        s = run_data[mode]['summary']
        row = {
            'Mode': mode_labels.get(mode, mode),
            'Images searched': s.get('images_searched', '?'),
            'Bound (bits/image)': s.get('bound_bits_per_image', '?'),
            'Aux bits (fixed)': s.get('aux_bits_fixed', '?'),
            'Aux bits (tight)': s.get('aux_bits_tight', '?'),
            'Mean ACC (22 atk)': s['means'].get('mean_acc_22_attacks', '?'),
        }
        if 'mean_acc_all_attacks' in s['means']:
            row['Mean ACC (all)'] = s['means']['mean_acc_all_attacks']
        # Mean images per message size
        for size, count in s.get('capacity_mean_images', {}).items():
            row[f'Imgs for {size}B'] = count
        capacity_rows.append(row)

    cap_df = pd.DataFrame(capacity_rows)
    display(cap_df)

### 8f — Retrieval accuracy summary chart

In [ ]:
"""Summary bar chart: overall mean accuracy for each retrieval mode.

This is the headline number — Phase 3 achieves 100% across all attacks.
"""

if not run_data:
    print("No runs loaded. Skipping.")
else:
    modes = sorted(run_data.keys())
    labels_list = [mode_labels.get(m, m) for m in modes]
    mean_accs = []
    for m in modes:
        s = run_data[m]['summary']['means']
        # Use all-attacks mean if available, else 22-attack mean
        acc = s.get('mean_acc_all_attacks', s.get('mean_acc_22_attacks', 0))
        mean_accs.append(acc)

    fig, ax = plt.subplots(figsize=(10, 5))
    bars_colors = [mode_colors.get(m, '#95A5A6') for m in modes]
    bars = ax.barh(range(len(modes)), mean_accs, color=bars_colors, alpha=0.85)

    for i, (acc, label) in enumerate(zip(mean_accs, labels_list)):
        ax.text(acc + 0.3, i, f'{acc:.1f}%', va='center', fontsize=11, fontweight='bold')

    ax.set_yticks(range(len(modes)))
    ax.set_yticklabels(labels_list, fontsize=10)
    ax.set_xlabel('Mean Accuracy (%)', fontsize=12)
    ax.set_xlim(0, 105)
    ax.axvline(x=100, color='#2ECC71', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title('Overall Mean Accuracy by Retrieval Mode', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 8g — Speed comparison

In [ ]:
"""Compare runtime (seconds) for each retrieval mode.

sscd_align is much slower than sscd (Fourier-Mellin alignment is expensive),
but on VOC2012 both achieve 100%, so the extra cost buys nothing.
"""

if not run_data:
    print("No runs loaded. Skipping.")
else:
    modes = sorted(run_data.keys())
    times = [run_data[m]['summary'].get('seconds', 0) for m in modes]
    labels_list = [mode_labels.get(m, m) for m in modes]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars_colors = [mode_colors.get(m, '#95A5A6') for m in modes]
    ax.barh(range(len(modes)), times, color=bars_colors, alpha=0.85)

    for i, (t, label) in enumerate(zip(times, labels_list)):
        ax.text(t + 2, i, f'{t:.0f}s ({t/60:.1f}m)', va='center', fontsize=10)

    ax.set_yticks(range(len(modes)))
    ax.set_yticklabels(labels_list, fontsize=10)
    ax.set_xlabel('Runtime (seconds)', fontsize=12)
    ax.set_title('Runtime Comparison by Retrieval Mode', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 8h — Extended attacks: SSCD is "unbreakable" on VOC2012

In [ ]:
"""Show extended attack results (if available).

All 34 attacks at 100% accuracy proves that SSCD copy detection
is extremely robust for VOC2012's 11,540 images.
The breaking point would require a much larger database (COCO: 164k images)
or adversarial attacks against the SSCD embedding space.
"""

ext_modes = [m for m in run_data if '_ext' in m]

if not ext_modes:
    print("No extended attack runs found. These may still be running on Kaggle.")
    print("Once complete, re-run this cell with the results attached.")
else:
    for mode in ext_modes:
        df = run_data[mode]['attacks']
        s = run_data[mode]['summary']
        means = s['means']

        print(f"\n{'=' * 60}")
        print(f"  {mode_labels.get(mode, mode)}")
        print(f"{'=' * 60}")
        print(f"  Mean ACC (22 original): {means.get('mean_acc_22_attacks', '?')}%")
        print(f"  Mean ACC (combined 6):  {means.get('mean_acc_combined', '?')}%")
        print(f"  Mean ACC (heavy 6):     {means.get('mean_acc_heavy', '?')}%")
        print(f"  Mean ACC (all 34):      {means.get('mean_acc_all_attacks', '?')}%")

        # Show per-attack for extended attacks only
        ext_attacks = list(EXTENDED_ORDER)
        ext_rows = df[df['attack'].isin(ext_attacks)]
        if not ext_rows.empty:
            print(f"\n  Extended attack details:")
            for _, row in ext_rows.iterrows():
                tag = '[COMBINED]' if row['attack'] in COMBINED else '[HEAVY]  '
                print(f"    {tag} {row['attack']:45s} ACC: {row['acc_percent']:.1f}%")

    # Summary
    print(f"\n{'─' * 60}")
    print("CONCLUSION: SSCD achieves 100% accuracy on ALL 34 attacks")
    print("on VOC2012 (11,540 images). The copy-detection embeddings")
    print("are robust enough to survive every attack we tested.")
    print("\nBreaking SSCD would require:")
    print("  • A much larger database (COCO: 164k images)")
    print("  • Or adversarial attacks against the embedding space")
    print(f"{'─' * 60}")

### 8i — Retrievability analysis

How many images survive the retrievability pre-check at different thresholds?

In [ ]:
"""Show retrievability analysis from runs that include it.

The retrievability check attacks each image with all 22 Table IX attacks
and measures what fraction of the time SSCD finds it again.
Images passing the threshold are 'usable' for sending.

Since SSCD gets 100% on all attacks, almost all images pass even at
threshold=1.0, so the capacity cost of being cautious is near zero.
"""

retrievability_data = {}
for mode, data in run_data.items():
    retr = data['summary'].get('retrievability')
    if retr and 'by_threshold' in retr:
        retrievability_data[mode] = retr

if not retrievability_data:
    print("No retrievability data found in the loaded runs.")
    print("This data is generated by runs with usable_threshold > 0.")
else:
    for mode, retr in retrievability_data.items():
        print(f"\n{mode_labels.get(mode, mode)}:")
        print(f"  Sendable images (with objects): {retr['sendable_images']}")
        print(f"  Usable at threshold {retr['usable_threshold']}: {retr['usable_images']}")
        print(f"  Groups with a usable image: {retr.get('groups_with_a_usable_image', '?')}")

        thresholds = retr['by_threshold']
        thresh_values = sorted(thresholds.keys(), key=float)
        usable_counts = [thresholds[t]['usable_images'] for t in thresh_values]

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(thresh_values, usable_counts, color='#3498DB', alpha=0.85)
        ax.set_xlabel('Usable Threshold', fontsize=11)
        ax.set_ylabel('Usable Images', fontsize=11)
        ax.set_title(f'Retrievability: Usable Images by Threshold ({mode_labels.get(mode, mode)})',
                     fontsize=12, fontweight='bold')

        # Add count labels on bars
        for x, y in zip(thresh_values, usable_counts):
            ax.text(x, y + 50, str(y), ha='center', fontsize=10, fontweight='bold')

        ax.axhline(y=retr['sendable_images'], color='#E74C3C', linestyle='--',
                   linewidth=1, alpha=0.7, label=f"Sendable: {retr['sendable_images']}")
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

---
## Summary

This notebook demonstrated the **complete coverless steganography pipeline**:

1. **Object Detection**: Faster R-CNN detects objects → top-M objects per image selected by area
2. **Dictionary**: DP-balanced mapping of object labels to n-bit binary groups
3. **Hiding**: Secret message segmented into n-bit groups → greedy image selection
4. **Attacks**: 34 attacks (geometric + non-geometric + combined + heavy) applied in the channel
5. **Retrieval**: SSCD copy-detection search finds the original despite attacks
6. **Decoding**: Recovered image's objects → groups → bits → original message


7. **Stress Testing**: Truly heavy attacks, strength sweeps to find breaking points,
   novel attacks (perspective warp, print-scan), and distractor images

**Key finding**: Phase 3 (SSCD retrieval) achieves **100% accuracy on all 34 attacks** on VOC2012,
but Section 9 reveals the actual limits of this result.

---
## Section 9 — Stress-Testing the 100% Claim

SSCD achieves 100% accuracy on all 34 attacks against VOC2012. But **is that
number meaningful?** This section puts it under real scrutiny:

1. **Plain SSCD (no retrievability filter)** — does the filter cherry-pick easy images?
2. **Truly heavy attacks** — the "heavy" attacks above use mild parameters.
   What happens with real-world noise variance 0.01–0.05, salt-and-pepper 5–10%,
   removing 50–75% of the image?
3. **Strength sweep** — sweep each attack parameter until retrieval fails.
   Where exactly does SSCD break?
4. **Novel attacks** — perspective warps, aspect-ratio changes, and print-scan
   simulation that SSCD was never trained to handle.
5. **Distractor images** — retrieval from 11k images is easy.
   What happens when we add 50k+ distractors from a different dataset?

> **Note on noise parameters:** The `gauss_noise(sigma)` function uses
> **standard deviation** (σ), not variance (σ²). So σ=0.1 means variance=0.01.
> The paper's "Gauss Noise" uses σ=0.001 (variance ~10⁻⁶), which is
> almost imperceptible. This section tests up to σ=0.3 (variance=0.09).

### 9a — Stress-test settings

How many images to test at each strength level.
More images = more reliable results but slower.

In [ ]:
"""Stress-test settings. Adjust these to trade speed for thoroughness."""

# Number of sample images to sweep over (more = slower but more reliable)
STRESS_N_IMAGES = 20

# SSCD similarity threshold — below this, retrieval is considered failed
SSCD_THRESHOLD = 0.5

# Pick sample images spread across the database
stress_rng = np.random.default_rng(42)
stress_indices = stress_rng.choice(
    np.flatnonzero(np.array([len(s) for s in index.sequences]) >= 1),
    size=min(STRESS_N_IMAGES, len(ids)),
    replace=False
)
stress_images = [read_image(image_paths[ids[i]]) for i in stress_indices]
stress_descs = descriptors[stress_indices]

print(f"Stress test: {len(stress_indices)} sample images selected")

### 9b — Truly heavy attacks

The current "heavy" attacks use mild parameters compared to real-world abuse.
Here we define **genuinely destructive** attacks:

| Attack | Paper value | Current "heavy" | Truly heavy |
|--------|-------------|-----------------|-------------|
| Gauss noise σ | 0.001 | 0.01 | 0.1–0.3 |
| Salt & pepper | 0.1% | 1% | 5–10% |
| Center crop | 5–20% | — | 50–75% |
| JPEG quality | 90 | 10 | 3–5 |
| Rotation | 10–50° | — | 90–180° |

In [ ]:
"""Define and apply truly heavy attacks.

These are far more destructive than anything in the paper or the
current extended set. Most would make images barely recognizable.
"""
from stego.attacks import (
    center_crop, edge_crop, rotate, translate, scale,
    gauss_noise, salt_and_pepper, speckle_noise, jpeg,
    histogram_equalization, gamma_correction
)

truly_heavy = {
    # Noise — variance 0.01 to 0.09 (sigma 0.1 to 0.3)
    "Gauss σ=0.10 (var=0.01)":  lambda img, rng: gauss_noise(img, 0.10, rng),
    "Gauss σ=0.15 (var=0.02)":  lambda img, rng: gauss_noise(img, 0.15, rng),
    "Gauss σ=0.22 (var=0.05)":  lambda img, rng: gauss_noise(img, 0.224, rng),
    "Gauss σ=0.30 (var=0.09)":  lambda img, rng: gauss_noise(img, 0.30, rng),
    # Salt & pepper 5–10%
    "S&P 5%":                   lambda img, rng: salt_and_pepper(img, 0.05, rng),
    "S&P 10%":                  lambda img, rng: salt_and_pepper(img, 0.10, rng),
    "S&P 20%":                  lambda img, rng: salt_and_pepper(img, 0.20, rng),
    # Cropping 50–75%
    "Center Crop 50%":          lambda img, rng: center_crop(img, 0.50),
    "Center Crop 75%":          lambda img, rng: center_crop(img, 0.75),
    "Edge Crop 50%":            lambda img, rng: edge_crop(img, 0.50),
    "Edge Crop 75%":            lambda img, rng: edge_crop(img, 0.75),
    # Extreme JPEG
    "JPEG Q5":                  lambda img, rng: jpeg(img, 5),
    "JPEG Q3":                  lambda img, rng: jpeg(img, 3),
    # Extreme rotation
    "Rotation 90°":             lambda img, rng: rotate(img, 90),
    "Rotation 135°":            lambda img, rng: rotate(img, 135),
    "Rotation 180°":            lambda img, rng: rotate(img, 180),
    # Resize to tiny then back (real crop-and-resize)
    "Resize 0.25× + back":     lambda img, rng: cv2.resize(
                                    cv2.resize(img, None, fx=0.25, fy=0.25),
                                    (img.shape[1], img.shape[0])),
    "Resize 0.10× + back":     lambda img, rng: cv2.resize(
                                    cv2.resize(img, None, fx=0.10, fy=0.10),
                                    (img.shape[1], img.shape[0])),
}

# Apply each to the demo image and compute SSCD similarity
print(f"{'Attack':40s}  {'SSCD':>6s}  {'Status':>8s}")
print("─" * 60)

heavy_scores = {}
for name, fn in truly_heavy.items():
    rng_h = attack_rng(config.seed, name, 0, 0)
    attacked = fn(demo_image, rng_h)
    atk_desc = sscd_model.describe([attacked])[0]
    score = float(demo_descriptor @ atk_desc)
    heavy_scores[name] = score
    status = "✓ PASS" if score > SSCD_THRESHOLD else "✗ FAIL"
    print(f"  {name:38s}  {score:6.4f}  {status}")

In [ ]:
"""Show the truly heavy attacks as an image grid."""

heavy_names = list(truly_heavy.keys())
n_h = len(heavy_names)
cols = 4
rows = (n_h + 1 + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
axes = axes.flatten()

axes[0].imshow(bgr_to_rgb(demo_image))
axes[0].set_title("ORIGINAL", fontsize=10, fontweight="bold")
axes[0].axis("off")

for i, name in enumerate(heavy_names):
    ax = axes[i + 1]
    rng_h = attack_rng(config.seed, name, 0, 0)
    attacked = truly_heavy[name](demo_image, rng_h)
    ax.imshow(bgr_to_rgb(attacked))
    score = heavy_scores[name]
    color = "#27AE60" if score > SSCD_THRESHOLD else "#E74C3C"
    ax.set_title(f"{name}\nSSCD: {score:.4f}", fontsize=7, color=color, fontweight="bold")
    ax.axis("off")

for j in range(n_h + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Truly Heavy Attacks — Where Does SSCD Break?",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 9c — Strength sweep: find the breaking point

Sweep each attack parameter from mild to extreme across multiple images.
For each strength, we measure the **mean SSCD similarity** and the
**retrieval success rate** (% of images where rank-1 match is correct).

In [ ]:
"""Sweep attack strength across multiple images.

For each attack family, sweep the parameter and record:
- Mean SSCD cosine similarity
- Retrieval success rate (rank-1 = correct original)
"""

sweep_configs = {
    "Rotation (°)": {
        "values": [0, 5, 10, 20, 30, 45, 60, 90, 120, 150, 180],
        "fn": lambda img, val, rng: rotate(img, val),
    },
    "Center Crop (%)": {
        "values": [0, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90],
        "fn": lambda img, val, rng: center_crop(img, val / 100.0) if val > 0 else img.copy(),
    },
    "JPEG Quality": {
        "values": [95, 90, 70, 50, 30, 20, 10, 7, 5, 3],
        "fn": lambda img, val, rng: jpeg(img, val),
    },
    "Gauss Noise σ": {
        "values": [0, 0.001, 0.005, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3],
        "fn": lambda img, val, rng: gauss_noise(img, val, rng) if val > 0 else img.copy(),
    },
    "S&P Density (%)": {
        "values": [0, 0.1, 0.5, 1, 2, 5, 10, 15, 20, 30],
        "fn": lambda img, val, rng: salt_and_pepper(img, val / 100.0, rng) if val > 0 else img.copy(),
    },
    "Edge Crop (%)": {
        "values": [0, 5, 10, 20, 30, 40, 50, 60, 70, 80],
        "fn": lambda img, val, rng: edge_crop(img, val / 100.0) if val > 0 else img.copy(),
    },
}

sweep_results = {}

for attack_name, cfg in sweep_configs.items():
    print(f"Sweeping {attack_name}...", end=" ", flush=True)
    results_for_attack = []

    for val in cfg["values"]:
        similarities = []
        retrievals_correct = 0

        for img_i, (img, orig_desc, orig_idx) in enumerate(
            zip(stress_images, stress_descs, stress_indices)
        ):
            rng_sw = np.random.default_rng([42, img_i, hash(attack_name) % 2**31, int(val * 1000)])
            attacked = cfg["fn"](img, val, rng_sw)
            atk_desc = sscd_model.describe([attacked])[0]
            cos_sim = float(orig_desc @ atk_desc)
            similarities.append(cos_sim)

            # Check rank-1 retrieval
            all_cos = retriever.descriptors @ atk_desc
            best_db_idx = int(np.argmax(all_cos))
            best_image_idx = int(retriever.search[best_db_idx])
            if best_image_idx == orig_idx:
                retrievals_correct += 1

        results_for_attack.append({
            "value": val,
            "mean_sim": np.mean(similarities),
            "std_sim": np.std(similarities),
            "min_sim": np.min(similarities),
            "retrieval_pct": 100.0 * retrievals_correct / len(stress_images),
        })

    sweep_results[attack_name] = results_for_attack
    print("done")

print("\nAll sweeps complete.")

In [ ]:
"""Plot the breaking-point curves for each attack family.

Two rows:
- Top: Mean SSCD similarity vs. attack strength (with error bands)
- Bottom: Retrieval success rate vs. attack strength
"""

n_sweeps = len(sweep_results)
fig, axes = plt.subplots(2, n_sweeps, figsize=(4.5 * n_sweeps, 8), squeeze=False)

for col, (attack_name, results) in enumerate(sweep_results.items()):
    values = [r["value"] for r in results]
    means = [r["mean_sim"] for r in results]
    stds = [r["std_sim"] for r in results]
    mins = [r["min_sim"] for r in results]
    retrieval = [r["retrieval_pct"] for r in results]

    x_labels = [str(v) for v in values]

    # Top row: SSCD similarity
    ax_top = axes[0, col]
    ax_top.plot(range(len(values)), means, "o-", color="#3498DB", linewidth=2, label="Mean")
    ax_top.fill_between(range(len(values)),
                        [m - s for m, s in zip(means, stds)],
                        [m + s for m, s in zip(means, stds)],
                        alpha=0.2, color="#3498DB")
    ax_top.plot(range(len(values)), mins, "v--", color="#E74C3C", linewidth=1, alpha=0.7, label="Min")
    ax_top.axhline(y=SSCD_THRESHOLD, color="#E74C3C", linestyle=":", linewidth=1.5, alpha=0.8)
    ax_top.set_xticks(range(len(values)))
    ax_top.set_xticklabels(x_labels, rotation=45, fontsize=7)
    ax_top.set_title(attack_name, fontsize=10, fontweight="bold")
    ax_top.set_ylim(-0.1, 1.05)
    if col == 0:
        ax_top.set_ylabel("SSCD Cosine Similarity", fontsize=10)
        ax_top.legend(fontsize=8)

    # Bottom row: Retrieval success
    ax_bot = axes[1, col]
    bar_colors = ["#27AE60" if r == 100 else "#E67E22" if r >= 50 else "#E74C3C" for r in retrieval]
    ax_bot.bar(range(len(values)), retrieval, color=bar_colors, alpha=0.85)
    ax_bot.axhline(y=100, color="#27AE60", linestyle="--", linewidth=1, alpha=0.5)
    ax_bot.set_xticks(range(len(values)))
    ax_bot.set_xticklabels(x_labels, rotation=45, fontsize=7)
    ax_bot.set_ylim(0, 110)
    if col == 0:
        ax_bot.set_ylabel("Retrieval Success (%)", fontsize=10)
    ax_bot.set_xlabel(attack_name.split("(")[-1].rstrip(")") if "(" in attack_name
                       else attack_name.split()[-1], fontsize=9)

fig.suptitle(f"Breaking-Point Analysis: Sweep Attack Strength (n={len(stress_images)} images)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print the breaking points
print("\nBreaking points (where retrieval first drops below 100%):")
print("─" * 65)
for attack_name, results in sweep_results.items():
    broke_at = None
    for r in results:
        if r["retrieval_pct"] < 100:
            broke_at = r
            break
    if broke_at:
        print(f"  {attack_name:25s}  breaks at value={broke_at['value']}"
              f"  (retrieval={broke_at['retrieval_pct']:.0f}%, SSCD={broke_at['mean_sim']:.4f})")
    else:
        print(f"  {attack_name:25s}  survives ALL tested values!")

### 9d — Novel attacks SSCD wasn't trained on

SSCD was trained with random resized crops and horizontal flips.
These attacks go beyond that training distribution:

- **Perspective warp**: simulates viewing angle changes (keystoning)
- **Aspect-ratio change**: stretch width or height independently
- **Print-scan simulation**: blur + noise + slight rotation + JPEG (mimics printing and re-scanning)
- **Colour channel swap**: BGR → RGB swap or channel shuffle
- **Extreme downscale**: reduce to thumbnail and enlarge back

In [ ]:
"""Implement novel attacks that SSCD wasn't trained to handle."""

def perspective_warp(image, strength, rng):
    """Simulates viewing angle change (keystoning effect).
    strength: fraction of image dimension for corner offsets (0.05 = 5%).
    """
    h, w = image.shape[:2]
    offset = int(max(h, w) * strength)
    # Random corner offsets
    dx = rng.integers(-offset, offset + 1, size=4)
    dy = rng.integers(-offset, offset + 1, size=4)
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([
        [dx[0], dy[0]], [w + dx[1], dy[1]],
        [w + dx[2], h + dy[2]], [dx[3], h + dy[3]]
    ])
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(image, M, (w, h), borderValue=(0, 0, 0))


def aspect_ratio_change(image, x_scale, y_scale):
    """Stretch width and height independently, then pad/crop back to original size."""
    h, w = image.shape[:2]
    new_w, new_h = int(w * x_scale), int(h * y_scale)
    stretched = cv2.resize(image, (new_w, new_h))
    # Resize back to original dimensions (distorting aspect ratio)
    return cv2.resize(stretched, (w, h))


def print_scan_sim(image, rng):
    """Simulate printing and re-scanning:
    1. Slight blur (printer dots)
    2. Add scanner noise
    3. Tiny rotation (paper skew)
    4. JPEG compression (scanner output)
    5. Contrast/brightness shift
    """
    # 1. Blur
    out = cv2.GaussianBlur(image, (3, 3), 1.0)
    # 2. Scanner noise
    out = gauss_noise(out, 0.02, rng)
    # 3. Slight rotation (±3°)
    angle = rng.uniform(-3, 3)
    out = rotate(out, angle)
    # 4. JPEG
    out = jpeg(out, 70)
    # 5. Brightness/contrast shift
    alpha = rng.uniform(0.85, 1.15)  # contrast
    beta = rng.uniform(-15, 15)      # brightness
    out = np.clip(out.astype(np.float64) * alpha + beta, 0, 255).astype(np.uint8)
    return out


def color_channel_swap(image, rng):
    """Swap colour channels randomly."""
    channels = list(rng.permutation(3))
    return image[:, :, channels]


def extreme_downscale(image, factor):
    """Downscale to tiny size and upscale back — destroys fine detail."""
    h, w = image.shape[:2]
    tiny = cv2.resize(image, (max(1, int(w * factor)), max(1, int(h * factor))),
                      interpolation=cv2.INTER_AREA)
    return cv2.resize(tiny, (w, h), interpolation=cv2.INTER_LINEAR)


novel_attacks = {
    "Perspective 5%":         lambda img, rng: perspective_warp(img, 0.05, rng),
    "Perspective 10%":        lambda img, rng: perspective_warp(img, 0.10, rng),
    "Perspective 20%":        lambda img, rng: perspective_warp(img, 0.20, rng),
    "Aspect 1.5×0.67":        lambda img, rng: aspect_ratio_change(img, 1.5, 0.67),
    "Aspect 0.5×2.0":         lambda img, rng: aspect_ratio_change(img, 0.5, 2.0),
    "Aspect 2.0×0.5":         lambda img, rng: aspect_ratio_change(img, 2.0, 0.5),
    "Print-Scan Sim":         lambda img, rng: print_scan_sim(img, rng),
    "Channel Swap":           lambda img, rng: color_channel_swap(img, rng),
    "Downscale 0.10× + back": lambda img, rng: extreme_downscale(img, 0.10),
    "Downscale 0.05× + back": lambda img, rng: extreme_downscale(img, 0.05),
}

print(f"Defined {len(novel_attacks)} novel attacks.")

In [ ]:
"""Apply novel attacks to multiple images and report SSCD similarity + retrieval."""

print(f"{'Attack':30s}  {'Mean SSCD':>9s}  {'Min SSCD':>8s}  {'Retrieval':>9s}")
print("─" * 65)

novel_results = {}

for name, fn in novel_attacks.items():
    sims = []
    correct = 0
    for img_i, (img, orig_desc, orig_idx) in enumerate(
        zip(stress_images, stress_descs, stress_indices)
    ):
        rng_n = np.random.default_rng([99, img_i, hash(name) % 2**31])
        attacked = fn(img, rng_n)
        atk_desc = sscd_model.describe([attacked])[0]
        cos_sim = float(orig_desc @ atk_desc)
        sims.append(cos_sim)

        all_cos = retriever.descriptors @ atk_desc
        best_db_idx = int(np.argmax(all_cos))
        best_image_idx = int(retriever.search[best_db_idx])
        if best_image_idx == orig_idx:
            correct += 1

    ret_pct = 100.0 * correct / len(stress_images)
    novel_results[name] = {
        "mean_sim": np.mean(sims), "min_sim": np.min(sims),
        "std_sim": np.std(sims), "retrieval_pct": ret_pct,
    }
    status = "✓" if ret_pct == 100 else "✗" if ret_pct < 50 else "△"
    print(f"  {name:28s}  {np.mean(sims):9.4f}  {np.min(sims):8.4f}  {ret_pct:8.1f}% {status}")

In [ ]:
"""Show novel attacks as an image grid with SSCD scores."""

novel_names = list(novel_attacks.keys())
n_nv = len(novel_names)
cols = 4
rows = (n_nv + 1 + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
axes = axes.flatten()

axes[0].imshow(bgr_to_rgb(demo_image))
axes[0].set_title("ORIGINAL", fontsize=10, fontweight="bold")
axes[0].axis("off")

for i, name in enumerate(novel_names):
    ax = axes[i + 1]
    rng_nv = np.random.default_rng([99, 0, hash(name) % 2**31])
    attacked = novel_attacks[name](demo_image, rng_nv)
    ax.imshow(bgr_to_rgb(attacked))
    r = novel_results[name]
    color = "#27AE60" if r["retrieval_pct"] == 100 else "#E74C3C"
    ax.set_title(f"{name}\nSSCD={r['mean_sim']:.3f}, Ret={r['retrieval_pct']:.0f}%",
                 fontsize=7, color=color, fontweight="bold")
    ax.axis("off")

for j in range(n_nv + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Novel Attacks SSCD Wasn't Trained On",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 9e — Distractor images: does database size matter?

Retrieval from 11,540 images is relatively easy. In a real system the
search set could be millions of images. We simulate this by **injecting
synthetic distractor descriptors** into the SSCD search index.

If no external dataset (e.g. COCO) is attached, we generate random unit
vectors as distractors — these are the *easiest* possible distractors
since real images would share more structure with the targets. If
retrieval fails with random vectors, it will certainly fail with real ones.

In [ ]:
"""Test how retrieval accuracy degrades as the database grows.

We progressively add distractor descriptors and re-test rank-1 retrieval
for the demo stego images under the DEMO_ATTACK.

Distractors are either:
1. Real COCO descriptors (if attached)
2. Random unit vectors on the 512-d sphere (worst-case easy baseline)
"""

# Check for COCO descriptors
coco_desc_path = None
for pattern in ["/kaggle/input/**/coco*_sscd*.npz",
                "/kaggle/input/**/coco*_descriptors*.npy"]:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        coco_desc_path = matches[0]
        break

distractor_source = "random"
if coco_desc_path:
    try:
        if coco_desc_path.endswith(".npz"):
            coco_descs = np.load(coco_desc_path)["descriptors"].astype(np.float32)
        else:
            coco_descs = np.load(coco_desc_path).astype(np.float32)
        # L2-normalize
        norms = np.linalg.norm(coco_descs, axis=1, keepdims=True)
        coco_descs = coco_descs / np.maximum(norms, 1e-8)
        distractor_source = f"COCO ({len(coco_descs)} images)"
        print(f"Loaded COCO descriptors: {coco_descs.shape}")
    except Exception as e:
        print(f"Could not load COCO descriptors: {e}")
        coco_descs = None
else:
    coco_descs = None

# Generate random distractors if needed
max_distractors = 200_000
dim = retriever.descriptors.shape[1]

if coco_descs is not None and len(coco_descs) >= max_distractors:
    all_distractors = coco_descs[:max_distractors]
else:
    # Random unit vectors
    rng_dist = np.random.default_rng(123)
    raw = rng_dist.standard_normal((max_distractors, dim)).astype(np.float32)
    norms = np.linalg.norm(raw, axis=1, keepdims=True)
    all_distractors = raw / norms
    if coco_descs is not None:
        # Use COCO first, then fill with random
        n_coco = len(coco_descs)
        all_distractors[:n_coco] = coco_descs
        distractor_source = f"COCO ({n_coco}) + random ({max_distractors - n_coco})"

print(f"Distractor source: {distractor_source}")
print(f"Distractor pool: {all_distractors.shape}")

In [ ]:
"""Sweep database size and measure retrieval accuracy.

At each size, we append distractors to the real VOC2012 descriptors
and test whether each stego image is still rank-1 under DEMO_ATTACK.
"""

db_sizes = [0, 1_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000]
db_sizes = [s for s in db_sizes if s <= len(all_distractors)]

original_db = retriever.descriptors  # shape (N_voc, 512)
original_search = retriever.search

# Prepare the attacked stego images once
stego_attacked = []
for img_num, item in enumerate(result.stego):
    orig = read_image(image_paths[ids[item.image_index]])
    rng_d = attack_rng(config.seed, DEMO_ATTACK, 0, img_num)
    attacked = all_attacks[DEMO_ATTACK](orig, rng_d)
    stego_attacked.append((attacked, item.image_index))

distractor_results = []

for n_distractors in db_sizes:
    # Build augmented database
    if n_distractors > 0:
        aug_db = np.vstack([original_db, all_distractors[:n_distractors]])
    else:
        aug_db = original_db

    total_size = len(aug_db)
    correct = 0

    for attacked_img, true_idx in stego_attacked:
        atk_desc = sscd_model.describe([attacked_img])[0]
        scores = aug_db @ atk_desc
        best_pos = int(np.argmax(scores))

        # Check if best match is in the original VOC range
        if best_pos < len(original_db):
            retrieved_idx = int(original_search[best_pos])
            if retrieved_idx == true_idx:
                correct += 1
        # If best match is a distractor, retrieval failed

    ret_pct = 100.0 * correct / len(stego_attacked)
    distractor_results.append({
        "n_distractors": n_distractors,
        "total_db": total_size,
        "retrieval_pct": ret_pct,
        "correct": correct,
        "total": len(stego_attacked),
    })
    status = "✓" if ret_pct == 100 else "✗"
    print(f"  DB size {total_size:>7,d} (+{n_distractors:>7,d} distractors): "
          f"{correct}/{len(stego_attacked)} correct = {ret_pct:.1f}% {status}")

In [ ]:
"""Plot retrieval accuracy vs. database size."""

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sizes = [r["total_db"] for r in distractor_results]
accs = [r["retrieval_pct"] for r in distractor_results]

# Left: accuracy vs DB size
bar_colors = ["#27AE60" if a == 100 else "#E67E22" if a >= 50 else "#E74C3C" for a in accs]
ax1.bar(range(len(sizes)), accs, color=bar_colors, alpha=0.85)
ax1.set_xticks(range(len(sizes)))
ax1.set_xticklabels([f"{s:,}" for s in sizes], rotation=45, fontsize=8)
ax1.set_xlabel("Total Database Size", fontsize=11)
ax1.set_ylabel("Retrieval Accuracy (%)", fontsize=11)
ax1.set_ylim(0, 110)
ax1.axhline(y=100, color="#27AE60", linestyle="--", linewidth=1, alpha=0.5)
ax1.set_title(f"Retrieval Under \"{DEMO_ATTACK}\" vs. DB Size", fontsize=12, fontweight="bold")

# Add labels
for i, (s, a) in enumerate(zip(sizes, accs)):
    ax1.text(i, a + 2, f"{a:.0f}%", ha="center", fontsize=10, fontweight="bold")

# Right: line chart for clarity
ax2.plot(sizes, accs, "o-", color="#3498DB", linewidth=2, markersize=8)
ax2.fill_between(sizes, accs, alpha=0.15, color="#3498DB")
ax2.set_xlabel("Total Database Size", fontsize=11)
ax2.set_ylabel("Retrieval Accuracy (%)", fontsize=11)
ax2.set_ylim(0, 110)
ax2.set_xscale("log")
ax2.axhline(y=100, color="#27AE60", linestyle="--", linewidth=1, alpha=0.5)
ax2.set_title("Log Scale: Accuracy Degradation", fontsize=12, fontweight="bold")
ax2.grid(True, alpha=0.3)

fig.suptitle(f"Distractor Test: {distractor_source}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 9f — Plain SSCD vs. filtered (retrievability check)

The retrievability pre-check tests each image against all 22 attacks.
Images that survive a certain fraction (`usable_threshold`) are "usable."
This could bias the 100% result — perhaps only easy images pass.

Here we compare the **plain `sscd` run** (no filter, all sendable images)
against the **usable0.9** run (only images that survived 90% of attacks).

In [ ]:
"""Compare plain SSCD vs filtered SSCD from experiment runs."""

plain_mode = "sscd"
filtered_modes = [m for m in run_data if "usable" in m and "ext" not in m]

if plain_mode not in run_data:
    print("No plain SSCD run found. The comparison requires a run with")
    print("retrieval=sscd and usable_threshold=0 (the default).")
elif not filtered_modes:
    print("No filtered SSCD run found. The comparison requires a run with")
    print("usable_threshold > 0 (e.g. usable0.9).")
else:
    # Both available — compare
    plain_df = run_data[plain_mode]["attacks"]
    plain_summary = run_data[plain_mode]["summary"]

    print(f"{'':40s}  {'Plain SSCD':>11s}", end="")
    for fm in filtered_modes:
        print(f"  {fm:>15s}", end="")
    print()
    print("─" * (55 + 17 * len(filtered_modes)))

    attack_order = list(TABLE_IX_ORDER)
    for attack_name in attack_order:
        plain_row = plain_df[plain_df["attack"] == attack_name]
        plain_acc = plain_row["acc_percent"].iloc[0] if not plain_row.empty else float("nan")
        print(f"  {attack_name:38s}  {plain_acc:10.1f}%", end="")
        for fm in filtered_modes:
            filt_df = run_data[fm]["attacks"]
            filt_row = filt_df[filt_df["attack"] == attack_name]
            filt_acc = filt_row["acc_percent"].iloc[0] if not filt_row.empty else float("nan")
            diff = filt_acc - plain_acc
            arrow = "↑" if diff > 0 else "↓" if diff < 0 else "="
            print(f"  {filt_acc:10.1f}% {arrow}", end="")
        print()

    # Summary
    plain_mean = plain_summary["means"].get("mean_acc_22_attacks", "?")
    print(f"\n  {'MEAN (22 attacks)':38s}  {plain_mean:>10}%", end="")
    for fm in filtered_modes:
        fm_mean = run_data[fm]["summary"]["means"].get("mean_acc_22_attacks", "?")
        print(f"  {fm_mean:>10}%  ", end="")
    print()

    # Image counts
    plain_imgs = plain_summary.get("images_searched", "?")
    print(f"\n  Images searched (plain): {plain_imgs}")
    for fm in filtered_modes:
        fm_imgs = run_data[fm]["summary"].get("images_searched", "?")
        retr = run_data[fm]["summary"].get("retrievability", {})
        sendable = retr.get("sendable_images", "?")
        print(f"  Images searched ({fm}): {fm_imgs} (from {sendable} sendable)")

### 9g — Stress-test summary

In [ ]:
"""Consolidate all stress-test findings into one summary chart."""

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Truly heavy attacks — pass/fail
ax1 = axes[0]
h_names = list(heavy_scores.keys())
h_scores = [heavy_scores[n] for n in h_names]
h_colors = ["#27AE60" if s > SSCD_THRESHOLD else "#E74C3C" for s in h_scores]
ax1.barh(range(len(h_names)), h_scores, color=h_colors, alpha=0.85)
ax1.axvline(x=SSCD_THRESHOLD, color="#E74C3C", linestyle=":", linewidth=2)
ax1.set_yticks(range(len(h_names)))
ax1.set_yticklabels(h_names, fontsize=7)
ax1.set_xlabel("SSCD Similarity", fontsize=10)
ax1.set_title("Truly Heavy Attacks", fontsize=11, fontweight="bold")
ax1.set_xlim(-0.1, 1.05)

# Panel 2: Novel attacks — retrieval rate
ax2 = axes[1]
nv_names = list(novel_results.keys())
nv_rets = [novel_results[n]["retrieval_pct"] for n in nv_names]
nv_colors = ["#27AE60" if r == 100 else "#E67E22" if r >= 50 else "#E74C3C" for r in nv_rets]
ax2.barh(range(len(nv_names)), nv_rets, color=nv_colors, alpha=0.85)
ax2.set_yticks(range(len(nv_names)))
ax2.set_yticklabels(nv_names, fontsize=7)
ax2.set_xlabel("Retrieval Accuracy (%)", fontsize=10)
ax2.set_title("Novel Attacks", fontsize=11, fontweight="bold")
ax2.set_xlim(0, 110)

# Panel 3: Distractor scaling
ax3 = axes[2]
d_sizes = [r["total_db"] for r in distractor_results]
d_accs = [r["retrieval_pct"] for r in distractor_results]
ax3.plot(d_sizes, d_accs, "o-", color="#3498DB", linewidth=2, markersize=8)
ax3.fill_between(d_sizes, d_accs, alpha=0.15, color="#3498DB")
ax3.set_xscale("log")
ax3.set_xlabel("Database Size", fontsize=10)
ax3.set_ylabel("Retrieval Accuracy (%)", fontsize=10)
ax3.set_title("Distractor Scaling", fontsize=11, fontweight="bold")
ax3.set_ylim(0, 110)
ax3.axhline(y=100, color="#27AE60", linestyle="--", linewidth=1, alpha=0.5)

fig.suptitle("Section 9 Summary: Where Does SSCD Actually Break?",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print consolidated findings
print("\n" + "═" * 70)
print("  STRESS TEST FINDINGS")
print("═" * 70)
n_heavy_pass = sum(1 for s in heavy_scores.values() if s > SSCD_THRESHOLD)
print(f"\n  Truly heavy attacks:  {n_heavy_pass}/{len(heavy_scores)} survived")
n_novel_pass = sum(1 for r in novel_results.values() if r["retrieval_pct"] == 100)
print(f"  Novel attacks:        {n_novel_pass}/{len(novel_results)} at 100% retrieval")

# Breaking points
print(f"\n  Breaking points:")
for attack_name, results in sweep_results.items():
    broke_at = None
    for r in results:
        if r["retrieval_pct"] < 100:
            broke_at = r
            break
    if broke_at:
        print(f"    {attack_name:25s}  breaks at {broke_at['value']}")
    else:
        print(f"    {attack_name:25s}  unbreakable in tested range")

# Distractor summary
for r in distractor_results:
    if r["retrieval_pct"] < 100:
        print(f"\n  Distractor limit: retrieval degrades at {r['total_db']:,} total images "
              f"(+{r['n_distractors']:,} distractors)")
        break
else:
    print(f"\n  Distractor limit: still 100% at {distractor_results[-1]['total_db']:,} total images")

print("═" * 70)

In [ ]:
print("Notebook complete. All sections finished.")